# YOLO11s + LoRA 独立正式训练

这是与项目主线隔离的正式 LoRA 训练方案，默认训练 150 epochs。权重、adapter、评估指标和 ONNX 全部保存在 `models/trained/notebook/` 与 `models/metrics/notebook_*`，不会覆盖主线或 smoke 产物。

In [1]:
from pathlib import Path
import json
import sys
from datetime import datetime, timezone

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'training').is_dir():
            return candidate
    raise FileNotFoundError('找不到项目根目录，请从项目内启动 Notebook')

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('项目根目录:', PROJECT_ROOT)

项目根目录: D:\dev\projects\yolo_train


In [2]:
import torch
import ultralytics
from ultralytics.cfg import DEFAULT_CFG_DICT

LORA_KEYS = ('lora_r', 'lora_alpha', 'lora_dropout', 'lora_gradient_checkpointing')
missing = [key for key in LORA_KEYS if key not in DEFAULT_CFG_DICT]
print('Python:', sys.version.split()[0])
print('Ultralytics / YOLO-Master:', ultralytics.__version__, ultralytics.__file__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if missing:
    raise RuntimeError(f'当前环境不是支持 LoRA 的 YOLO-Master，缺少: {missing}')
if not torch.cuda.is_available():
    raise RuntimeError('正式训练要求 CUDA；当前未检测到可用 GPU')

Python: 3.13.9
Ultralytics / YOLO-Master: 8.3.240 D:\dev\projects\yolo_train\vendor\YOLO-Master\ultralytics\__init__.py
CUDA: True
GPU: NVIDIA GeForce RTX 5060 Laptop GPU


In [3]:
from training.audit_dataset import audit

DATASET_DIR = PROJECT_ROOT / 'data/processed'
DATA_YAML = DATASET_DIR / 'data.yaml'
audit_report = audit(DATASET_DIR)
if not audit_report['passed']:
    raise RuntimeError('数据审计失败，禁止训练：\n' + '\n'.join(audit_report['errors'][:20]))
audit_report['splits']

{'train': {'images': 2816,
  'class_objects': {'crack': 170, 'water': 847, 'leak': 1154},
  'negative_images': 708,
  'objects_per_image': {0: 708, 1: 2067, 2: 30, 3: 3, 4: 5, 5: 3}},
 'valid': {'images': 251,
  'class_objects': {'crack': 18, 'water': 85, 'leak': 97},
  'negative_images': 53,
  'objects_per_image': {0: 53, 1: 196, 2: 2}},
 'test': {'images': 251,
  'class_objects': {'crack': 18, 'water': 84, 'leak': 102},
  'negative_images': 53,
  'objects_per_image': {0: 53, 1: 194, 2: 3, 4: 1}}}

## 正式训练配置

稳定版配方默认 150 epochs、batch=8、workers=0。它使用独立的 v2 目录，不覆盖已经暂停的旧训练。若 8GB 显存不足可把 batch 改为 4。`RESUME=True` 时只会从 v2 的 `last.pt` 继续。

In [4]:
MODEL = 'yolo11s.pt'
EPOCHS = 150
IMAGE_SIZE = 640
BATCH = 8
DEVICE = 0
SEED = 42
RESUME = False

OUTPUT_PROJECT = PROJECT_ROOT / 'models/trained/notebook'
RUN_NAME = 'yolo11s_lora_formal_v2'
RUN_DIR = OUTPUT_PROJECT / RUN_NAME
LAST_PT = RUN_DIR / 'weights/last.pt'
BEST_PT = RUN_DIR / 'weights/best.pt'
ADAPTER_DIR = RUN_DIR / 'lora_adapter'
METRICS_JSON = PROJECT_ROOT / 'models/metrics/notebook_lora_v2_internal_metrics.json'

if RESUME and not LAST_PT.is_file():
    raise FileNotFoundError(f'要求断点续训，但不存在: {LAST_PT}')
print('正式训练目录:', RUN_DIR)
print('断点续训:', RESUME)

正式训练目录: D:\dev\projects\yolo_train\models\trained\notebook\yolo11s_lora_formal_v2
断点续训: False


In [5]:
from ultralytics import YOLO

model = YOLO(str(LAST_PT) if RESUME else MODEL)
train_kwargs = dict(
    data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMAGE_SIZE, batch=BATCH, device=DEVICE, workers=0,
    optimizer='AdamW', lr0=0.001, lrf=0.01, weight_decay=0.0005, patience=20, seed=SEED,
    deterministic=True, amp=False, pretrained=True, cos_lr=True,
    warmup_epochs=5, warmup_bias_lr=0.01, save_period=10,
    lora_r=8, lora_alpha=8, lora_dropout=0.05, lora_freeze_bn=True,
    lora_use_rslora=False, lora_init_lora_weights='gaussian', lora_lr_mult=0.5,
    lora_alpha_warmup=0,
    lora_layer_decay=1.0, lora_ortho_weight=0.0,
    lora_dropout_end=0.05, lora_dropout_start_ratio=1.0,
    lora_gradient_checkpointing=False, lora_include_head=False,
    project=str(OUTPUT_PROJECT), name=RUN_NAME, exist_ok=True, plots=True, save=True,
)
if RESUME:
    train_kwargs['resume'] = True
train_results = model.train(**train_kwargs)

D:\dev\runtimes\conda\envs\yolo\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


New https://pypi.org/project/ultralytics/8.4.98 available  Update with 'pip install -U ultralytics'
Ultralytics 8.3.240  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cluster=False, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=D:\dev\projects\yolo_train\data\processed\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, iou_type=iou, keras=False, kobj=1.0, line_width=None, lora_adapter_dir=lora_adapter, lora_allow_depthwise=False, lora_allow_rtdetr_dora=False, lor

In [6]:
if not BEST_PT.is_file():
    raise FileNotFoundError(f'正式训练没有生成 best.pt: {BEST_PT}')
results_csv = RUN_DIR / 'results.csv'
if not results_csv.is_file():
    raise FileNotFoundError(f'训练没有生成 results.csv: {results_csv}')
import csv
with results_csv.open(encoding='utf-8') as handle:
    metric_rows = list(csv.DictReader(handle))
map50_values = [float(row['metrics/mAP50(B)']) for row in metric_rows]
best_map50 = max(map50_values, default=0.0)
if best_map50 < 0.05:
    raise RuntimeError(f'训练指标未通过最低稳定性检查，best mAP50={best_map50:.4f}，不要导出或部署该模型')
if hasattr(model, 'save_lora_only'):
    model.save_lora_only(str(ADAPTER_DIR))
print('正式 best.pt:', BEST_PT)
print('LoRA adapter:', ADAPTER_DIR)
print(f'训练稳定性检查通过，best mAP50={best_map50:.4f}')

[LoRA]  Adapters saved to D:\dev\projects\yolo_train\models\trained\notebook\yolo11s_lora_formal_v2\lora_adapter
正式 best.pt: D:\dev\projects\yolo_train\models\trained\notebook\yolo11s_lora_formal_v2\weights\best.pt
LoRA adapter: D:\dev\projects\yolo_train\models\trained\notebook\yolo11s_lora_formal_v2\lora_adapter
训练稳定性检查通过，best mAP50=0.8546


## 内部 test 正式评估
评估使用独立的 251 张 test，不使用训练或 valid 指标代替。

In [7]:
from training.common import TARGET_NAMES

formal_model = YOLO(str(BEST_PT))
metrics = formal_model.val(
    data=str(DATA_YAML), split='test', imgsz=640, batch=BATCH, workers=0, device=DEVICE, plots=True,
    project=str(PROJECT_ROOT / 'models/metrics/evaluation_runs'),
    name='notebook_lora_v2_internal', exist_ok=True,
)
maps = metrics.box.maps
metric_payload = {
    'source': 'formal_lora_notebook',
    'model_path': str(BEST_PT),
    'dataset': str(DATA_YAML),
    'split': 'test',
    'epochs': EPOCHS,
    'precision': float(metrics.box.mp),
    'recall': float(metrics.box.mr),
    'map50': float(metrics.box.map50),
    'map50_95': float(metrics.box.map),
    'per_class_map50_95': {name: float(maps[i]) for i, name in enumerate(TARGET_NAMES)},
    'speed_ms': {key: float(value) for key, value in metrics.speed.items()},
    'model_size_mb': BEST_PT.stat().st_size / (1024 * 1024),
    'created_at': datetime.now(timezone.utc).isoformat(),
}
METRICS_JSON.parent.mkdir(parents=True, exist_ok=True)
METRICS_JSON.write_text(json.dumps(metric_payload, ensure_ascii=False, indent=2), encoding='utf-8')
metric_payload

Ultralytics 8.3.240  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
[LoRA] Fusion disabled for LoRADetectionModel.
val: Fast image access  (ping: 0.10.1 ms, read: 118.930.6 MB/s, size: 53.9 KB)
val: Scanning D:\dev\projects\yolo_train\data\processed\test\labels.cache... 251 images, 53 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 251/251 493.6Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 32/32 8.8it/s 3.6s0.1s
                   all        251        204      0.908      0.784      0.889      0.798
                 crack         18         18      0.928      0.667      0.857      0.791
                 water         84         84      0.937      0.821      0.903      0.783
                  leak         97        102       0.86      0.863      0.908      0.818
Speed: 0.2ms preprocess, 7.4ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to D:\dev\projects\yolo_tr

{'source': 'formal_lora_notebook',
 'model_path': 'D:\\dev\\projects\\yolo_train\\models\\trained\\notebook\\yolo11s_lora_formal_v2\\weights\\best.pt',
 'dataset': 'D:\\dev\\projects\\yolo_train\\data\\processed\\data.yaml',
 'split': 'test',
 'epochs': 150,
 'precision': 0.908498243044804,
 'recall': 0.7836134453781513,
 'map50': 0.889384833636528,
 'map50_95': 0.7975087892746331,
 'per_class_map50_95': {'crack': 0.7914254975808503,
  'water': 0.7827278395446822,
  'leak': 0.8183730306983664},
 'speed_ms': {'preprocess': 0.18886135451855562,
  'inference': 7.4243868526768875,
  'loss': 0.0009466135648053244,
  'postprocess': 0.6764402391112584},
 'model_size_mb': 19.459050178527832,
 'created_at': '2026-07-17T13:24:10.560083+00:00'}

## 独立 ONNX 导出（可选）
ONNX 保存在 Notebook 自己的目录，不写入 `models/deployed/`。

In [8]:
EXPORT_ONNX = True
if EXPORT_ONNX:
    exported_path = formal_model.export(format='onnx', imgsz=640, device=DEVICE)
    print('ONNX:', exported_path)
else:
    print('已跳过 ONNX 导出')

Export preflight: format=onnx, routed_modules=0, strategies=[]
Ultralytics 8.3.240  Python-3.13.9 torch-2.13.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Laptop GPU, 8151MiB)
 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
[LoRA] Fusion disabled for LoRADetectionModel.

PyTorch: starting from 'D:\dev\projects\yolo_train\models\trained\notebook\yolo11s_lora_formal_v2\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 7, 8400) (19.5 MB)
requirements: Ultralytics requirement ['onnxruntime-gpu'] not found, attempting AutoUpdate...
Using Python 3.13.9 environment at: D:\dev\runtimes\conda\envs\yolo
Resolved 5 packages in 2.43s
 Downloaded onnxruntime-gpu
Prepared 2 packages in 1m 17s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=co

In [9]:
manifest = {
    'training_line': 'notebook_formal_lora_v2_stable',
    'mainline_isolated': True,
    'best_pt': str(BEST_PT),
    'adapter_dir': str(ADAPTER_DIR),
    'metrics_json': str(METRICS_JSON),
    'eligible_for_mainline_comparison': True,
    'automatically_deployed': False,
    'recipe': 'freeze_bn_amp_off_low_warmup_stable_lora',
}
manifest_path = RUN_DIR / 'notebook_model_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
manifest

{'training_line': 'notebook_formal_lora_v2_stable',
 'mainline_isolated': True,
 'best_pt': 'D:\\dev\\projects\\yolo_train\\models\\trained\\notebook\\yolo11s_lora_formal_v2\\weights\\best.pt',
 'adapter_dir': 'D:\\dev\\projects\\yolo_train\\models\\trained\\notebook\\yolo11s_lora_formal_v2\\lora_adapter',
 'metrics_json': 'D:\\dev\\projects\\yolo_train\\models\\metrics\\notebook_lora_v2_internal_metrics.json',
 'eligible_for_mainline_comparison': True,
 'automatically_deployed': False,
 'recipe': 'freeze_bn_amp_off_low_warmup_stable_lora'}

## 纳入主线比较
本 Notebook 会产出正式模型，但不会自动部署。如果之后希望它参加最终客户模型选择，应将 `notebook_lora_v2_internal_metrics.json` 与主线完整微调结果按同一选择规则比较，并明确执行部署导出。